# **Multi-experiment align images**
### Batch-aligns images of the same cells taken before and after fixation

In [ ]:
import pandas as pd
import numpy as np
import shutil
from pathlib import Path
from microscopy_analysis.d00_utils import utilities as utils
from microscopy_analysis.d00_utils import dirnames as dn
from microscopy_analysis.d01_init_proc import align_two_imgs

## **Reorganize file structure prior to alignment**

#### Starting directory structure:
-dir1<br>
-dir1_phalloidin<br>
-dir2<br>
-dir2_phalloidin<br>
-dir3<br>
-dir3_phalloidin<br>

#### Goal multi-experiment file directory structure:
-multiexp_dir<br>
---aligndir_1<br>
-----dir1<br>
-----dir1_phalloidin<br>
---aligndir_2<br>
-----dir2<br>
-----dir2_phalloidin<br>



In [ ]:
multiexp_dir = Path(input('Please enter a path for the multi-exp directory to be aligned:'))

In [ ]:
dirnames = [Path(dir).name for dir in multiexp_dir.iterdir() if dir.is_dir()]
dirnames.sort()
dirs_df = pd.DataFrame()
dirs_df['dirname'] = dirnames

In [ ]:
splits = dirs_df['dirname'].str.split('_')
dirs_df['experiment'] = splits.str[0]
dirs_df['DIV'] = splits.str[1]
dirs_df['dish'] = splits.str[2]

dirs_df['phalloidin'] = np.where(dirs_df['dirname'].str.contains('phalloidin'), '_phalloidin', '')
dirs_df['basename'] = dirs_df['experiment'] + '_' + dirs_df['DIV'] + '_' + dirs_df['dish']

dirs_df

In [ ]:
# make align directories containing directories to be aligned

align_dirs = dirs_df['dish'].unique()

for align_dir in align_dirs:
    align_dirpath = multiexp_dir / align_dir
    align_dirpath.mkdir(exist_ok=True)
    
    df_filt = dirs_df[dirs_df['dish']==align_dir]
    
    for i, row in df_filt.iterrows():
        orig_dirpath = multiexp_dir / row['dirname']
        if orig_dirpath.is_dir():
            shutil.move(orig_dirpath, align_dirpath)

In [ ]:
# Within each image, select channel to be used for alignment
img1_ch_align = 1
img2_ch_align = 1

# Select channels to retain once images are aligned
img1_ch_subset = [0, 1, 2]
img2_ch_subset = [3]

img2_dir_uniquestring='phal'

In [ ]:
align_two_imgs.multiexp_align(multiexp_dir=multiexp_dir, img1_ch_align=img1_ch_align, img2_ch_align=img2_ch_align, img1_ch_subset=img1_ch_subset, img2_ch_subset=img2_ch_subset, img2_dir_uniquestring=img2_dir_uniquestring)

# Reorganize files

In [ ]:
# Move all CZI files to 1 folder
new_czi_dirpath = multiexp_dir / dn.orig_unaligned_dirname / dn.CZI_dirname
new_czi_dirpath.mkdir(exist_ok=True)
for path in multiexp_dir.rglob('*.czi'):
    path.rename(new_czi_dirpath / path.name)

In [ ]:
# combine and save csv files
aligned_dirpath = multiexp_dir / dn.aligned_dirname

def combine_dfs(input_dirpath, csvname):
    combined_df = pd.DataFrame()
    for path in aligned_dirpath.rglob(csvname):
        indiv_df = pd.read_csv(path)
        combined_df = pd.concat([combined_df, indiv_df])
    return combined_df

comb_align_info_df = combine_dfs(aligned_dirpath, 'alignment_info.csv')
print(f'The combined alignment info df has {len(comb_align_info_df)} rows')
comb_err_info_df = combine_dfs(aligned_dirpath, 'alignment_errors.csv')
print(f'The error info df has {len(comb_err_info_df)} rows')

tables_dirpath = aligned_dirpath / dn.proc_dirname / dn.tables_dirname
tables_dirpath.mkdir(parents=True, exist_ok=True)

comb_align_info_df.to_csv(tables_dirpath /'alignment_info.csv', index=False)
comb_err_info_df.to_csv(tables_dirpath /'alignment_errors.csv', index=False)


In [ ]:
# move aligned files out of subdirectories

aligned_dirpath = multiexp_dir / dn.aligned_dirname

indiv_ometif_dirpaths = [path for path in aligned_dirpath.rglob(f'*{dn.raw_ometif_dirname}')]
indiv_ometif_dirpaths

indiv_ometif_chsubset_dirpaths = [path for path in aligned_dirpath.rglob(f'*{dn.raw_ometif_chsubset_dirname}*')]
indiv_ometif_chsubset_dirpaths

raw_ometif_dirpath = aligned_dirpath / dn.proc_dirname / dn.raw_ometif_dirname
raw_ometif_dirpath.mkdir(parents=True, exist_ok=True)

raw_ometif_chsubset_dirpath = aligned_dirpath / dn.proc_dirname / dn.raw_ometif_chsubset_dirname
raw_ometif_chsubset_dirpath.mkdir(parents=True, exist_ok=True)

for dirpath in indiv_ometif_dirpaths:
    for imgpath in dirpath.glob('*.ome.tif'):
        new_imgpath = raw_ometif_dirpath / imgpath.name
        imgpath.rename(new_imgpath)
        
for dirpath in indiv_ometif_chsubset_dirpaths:
    for imgpath in dirpath.glob('*.ome.tif'):
        new_imgpath = raw_ometif_chsubset_dirpath / imgpath.name
        imgpath.rename(new_imgpath)